# Running Analysis at Scale with Papermill

In this notebook, we'll learn how to:
1. Run a parameterized notebook across multiple stocks and date ranges
2. Collect and aggregate results using scrapbook
3. Build a summary report from all the individual analyses

This pattern is useful in finance when you need to run the same analysis across:
- A universe of stocks
- Multiple time periods (monthly, quarterly)
- Different parameter combinations (backtesting)

---

## How Papermill and Scrapbook Work

Before we dive in, let's understand what these libraries actually do under the hood—this isn't magic, it's clever engineering.

### Papermill: Executing Notebooks Programmatically

**What it does**: Papermill lets you run a Jupyter notebook from Python code, injecting parameters and saving the executed result.

**How it works**:
1. **Reading**: Papermill reads your template notebook as a JSON file (`.ipynb` files are just JSON with a specific structure containing cells, outputs, and metadata)
2. **Parameter injection**: It looks for a cell tagged with `parameters` in the cell metadata. It then creates a *new cell* immediately after it containing your custom parameter values, which override the defaults
3. **Execution**: It uses the Jupyter execution engine (`nbclient`) to run each cell in order, just like clicking "Run All" manually
4. **Saving**: The executed notebook (with all outputs) is saved to your specified output path

**Key insight**: Your template notebook runs in its own kernel, completely isolated. Each execution is independent—running AAPL analysis doesn't affect MSFT analysis.

```python
# Under the hood, this is roughly what happens:
# 1. nb = read_notebook('template.ipynb')  # Load as JSON
# 2. inject_parameters(nb, {'ticker': 'AAPL'})  # Add new cell
# 3. execute_notebook(nb)  # Run all cells via nbclient
# 4. write_notebook(nb, 'output.ipynb')  # Save with outputs
```

### Scrapbook: Persisting and Collecting Results

**What it does**: Scrapbook lets you save ("glue") values from inside a running notebook and later read them back from the saved notebook file.

**How it works**:
1. **Gluing (inside notebook)**: When you call `sb.glue('name', value)`, scrapbook serializes the value and stores it in the notebook's cell output as a special MIME type (`application/scrapbook.scrap+json`)
2. **Reading (outside notebook)**: When you call `sb.read_notebook('output.ipynb')`, scrapbook parses the notebook JSON, finds all outputs with the scrapbook MIME type, and deserializes them back into Python objects
3. **Encoders**: Different data types need different serialization. Simple types (int, float, str) use JSON. Pandas DataFrames use the `pandas` encoder (which converts to JSON records). You can even store matplotlib figures!

**Key insight**: The "scraps" are stored *inside* the notebook file itself. This means your results travel with your executed notebook—no separate database or files needed.

```python
# Inside template notebook:
sb.glue('total_return', 15.5)  # Stored in cell output as special JSON

# Later, reading results:
nb = sb.read_notebook('output.ipynb')
nb.scraps['total_return'].data  # Deserializes back to 15.5
```

### Why This Matters

Understanding these mechanisms helps you:
- **Debug issues**: If a parameter isn't being injected, check that your parameters cell is properly tagged
- **Design templates**: Know that each run is isolated—global state doesn't persist between runs
- **Choose what to glue**: Simple metrics are easy; large DataFrames work but increase file size
- **Understand limitations**: Notebooks must complete successfully for scraps to be saved

In [ ]:
import papermill as pm
import scrapbook as sb
import pandas as pd
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta
import os
import sqlite3

import matplotlib.pyplot as plt
plt.style.use('ggplot')
%matplotlib inline

## 1. Understanding the Problem

Let's say we want to analyze 10 stocks over the first half of 2024, producing a separate report for each stock. Without automation, we would need to:
1. Open the template notebook
2. Change the ticker parameter
3. Run all cells
4. Save the notebook
5. Repeat 9 more times

With papermill, we write a loop and let the computer do this work.

In [ ]:
# Configuration
TEMPLATE_NOTEBOOK = '3_stock_analysis_template.ipynb'
OUTPUT_FOLDER = 'outputs'
DB_PATH = 'data/data.db'

# Let's see what tickers are available in our database
conn = sqlite3.connect(DB_PATH)
available_tickers = pd.read_sql("SELECT DISTINCT ticker FROM ohlc ORDER BY ticker", conn)['ticker'].tolist()
conn.close()

print(f"Available tickers ({len(available_tickers)}):")
print(available_tickers)

In [ ]:
# Select a subset of stocks to analyze
# (You can modify this list to include any tickers from the database)
tickers_to_analyze = ['AAPL', 'MSFT', 'NVDA', 'JPM', 'JNJ', 'BA']

# Define the analysis period
start_date = '2024-01-01'
end_date = '2024-06-30'

print(f"Will analyze {len(tickers_to_analyze)} stocks: {tickers_to_analyze}")
print(f"Period: {start_date} to {end_date}")

## 2. Running Notebooks with Papermill

Papermill's `execute_notebook()` function:
- Takes an input notebook (template)
- Runs it with specified parameters
- Saves the executed notebook to an output path

The magic happens because our template has a cell tagged with `parameters`. Papermill injects a new cell right after it with our custom values.

In [ ]:
# Create output directory if it doesn't exist
if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)
    print(f"Created output folder: {OUTPUT_FOLDER}")
else:
    print(f"Output folder exists: {OUTPUT_FOLDER}")

In [ ]:
# First, let's run a single notebook to see how it works
demo_ticker = 'AAPL'

print(f"Running analysis for {demo_ticker}...")

output_path = f"{OUTPUT_FOLDER}/{demo_ticker}_analysis.ipynb"

pm.execute_notebook(
    TEMPLATE_NOTEBOOK,
    output_path,
    parameters={
        'ticker': demo_ticker,
        'start_date': start_date,
        'end_date': end_date,
        'db_path': DB_PATH,
        'volatility_window': 20
    }
)

print(f"Done! Output saved to: {output_path}")

## 3. Running Multiple Analyses

Now let's run the template for all our selected stocks.

In [ ]:
# Track successful and failed runs
successful_runs = []
failed_runs = []

for ticker in tickers_to_analyze:
    output_path = f"{OUTPUT_FOLDER}/{ticker}_analysis.ipynb"
    
    print(f"Processing {ticker}...", end=' ')
    
    try:
        pm.execute_notebook(
            TEMPLATE_NOTEBOOK,
            output_path,
            parameters={
                'ticker': ticker,
                'start_date': start_date,
                'end_date': end_date,
                'db_path': DB_PATH,
                'volatility_window': 20
            }
        )
        successful_runs.append(ticker)
        print("Done")
        
    except Exception as e:
        failed_runs.append((ticker, str(e)))
        print(f"FAILED: {e}")

print(f"\n=== Summary ===")
print(f"Successful: {len(successful_runs)}")
print(f"Failed: {len(failed_runs)}")

## 4. Collecting Results with Scrapbook

Each executed notebook contains "scraps" - values we saved with `sb.glue()` in the template. Scrapbook lets us read these values back without re-running the notebooks.

In [ ]:
# Read results from a single notebook
nb = sb.read_notebook(f"{OUTPUT_FOLDER}/AAPL_analysis.ipynb")

print("Available scraps:")
for name, scrap in nb.scraps.items():
    # Show value for simple types, just the type for complex ones
    if isinstance(scrap.data, (str, int, float, bool)):
        print(f"  {name}: {scrap.data}")
    else:
        print(f"  {name}: <{type(scrap.data).__name__}>")

In [ ]:
# Read all notebooks in the output folder
notebooks = sb.read_notebooks(OUTPUT_FOLDER)

print(f"Found {len(notebooks)} notebooks in {OUTPUT_FOLDER}")

## 5. Building a Summary Report

Now we'll aggregate the results from all notebooks into a single DataFrame for comparison.

In [ ]:
# Extract key metrics from each notebook
results = []

for path, scraps in notebooks.notebook_scraps.items():
    # Build a row of data from the scraps
    row = {
        'ticker': scraps['ticker'].data,
        'start_date': scraps['start_date'].data,
        'end_date': scraps['end_date'].data,
        'trading_days': scraps['trading_days'].data,
        'total_return_pct': scraps['total_return_pct'].data,
        'avg_volatility': scraps['avg_volatility'].data,
        'max_drawdown_pct': scraps['max_drawdown_pct'].data,
        'data_quality_ok': scraps['data_quality_ok'].data,
    }
    results.append(row)

# Create summary DataFrame
summary_df = pd.DataFrame(results)
summary_df = summary_df.sort_values('total_return_pct', ascending=False)
summary_df

In [ ]:
# Create a nicely formatted summary table
display_df = summary_df.copy()
display_df['total_return_pct'] = display_df['total_return_pct'].apply(lambda x: f"{x:.2f}%")
display_df['avg_volatility'] = display_df['avg_volatility'].apply(lambda x: f"{x:.1%}")
display_df['max_drawdown_pct'] = display_df['max_drawdown_pct'].apply(lambda x: f"{x:.2f}%")
display_df['data_quality_ok'] = display_df['data_quality_ok'].apply(lambda x: 'OK' if x else 'Issues')

display_df.columns = ['Ticker', 'Start', 'End', 'Days', 'Return', 'Avg Vol', 'Max DD', 'Quality']
print(f"\n=== Stock Performance Summary: {start_date} to {end_date} ===")
print(display_df.to_string(index=False))

## 6. Visualizing Comparative Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Sort by return for consistent ordering
plot_df = summary_df.sort_values('total_return_pct', ascending=True)

# Plot 1: Total Return
colors = ['green' if x > 0 else 'red' for x in plot_df['total_return_pct']]
axes[0].barh(plot_df['ticker'], plot_df['total_return_pct'], color=colors)
axes[0].axvline(x=0, color='black', linewidth=0.5)
axes[0].set_xlabel('Total Return (%)')
axes[0].set_title('Total Return by Stock')

# Plot 2: Average Volatility
plot_df_vol = summary_df.sort_values('avg_volatility', ascending=True)
axes[1].barh(plot_df_vol['ticker'], plot_df_vol['avg_volatility'] * 100, color='purple', alpha=0.7)
axes[1].set_xlabel('Average Volatility (%)')
axes[1].set_title('Average Volatility by Stock')

# Plot 3: Return vs Volatility (Risk-Reward)
axes[2].scatter(summary_df['avg_volatility'] * 100, summary_df['total_return_pct'], s=100)
for idx, row in summary_df.iterrows():
    axes[2].annotate(row['ticker'], 
                     (row['avg_volatility'] * 100, row['total_return_pct']),
                     xytext=(5, 5), textcoords='offset points', fontsize=9)
axes[2].axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
axes[2].set_xlabel('Average Volatility (%)')
axes[2].set_ylabel('Total Return (%)')
axes[2].set_title('Return vs Volatility')

plt.tight_layout()
plt.show()

## 7. Advanced: Running Across Multiple Time Periods

Often we want to analyze the same stocks across multiple time periods (e.g., monthly or quarterly). Here's how to set that up.

In [ ]:
# Generate quarterly periods for 2024
periods = [
    ('2024-01-01', '2024-03-31', 'Q1'),
    ('2024-04-01', '2024-06-30', 'Q2'),
    ('2024-07-01', '2024-09-30', 'Q3'),
    ('2024-10-01', '2024-12-06', 'Q4'),  # Data ends at 2024-12-06
]

# Select fewer stocks for this demo
multi_period_tickers = ['AAPL', 'MSFT', 'NVDA']

print(f"Will run {len(multi_period_tickers)} stocks x {len(periods)} periods = {len(multi_period_tickers) * len(periods)} analyses")

In [ ]:
# Create subfolder for quarterly results
quarterly_folder = f"{OUTPUT_FOLDER}/quarterly"
if not os.path.exists(quarterly_folder):
    os.makedirs(quarterly_folder)

# Run all combinations
for ticker in multi_period_tickers:
    for start, end, period_name in periods:
        output_path = f"{quarterly_folder}/{ticker}_{period_name}_analysis.ipynb"
        
        print(f"Processing {ticker} {period_name}...", end=' ')
        
        try:
            pm.execute_notebook(
                TEMPLATE_NOTEBOOK,
                output_path,
                parameters={
                    'ticker': ticker,
                    'start_date': start,
                    'end_date': end,
                    'db_path': DB_PATH,
                    'volatility_window': 20
                }
            )
            print("Done")
        except Exception as e:
            print(f"FAILED: {e}")

print("\nAll quarterly analyses complete!")

In [ ]:
# Collect quarterly results
quarterly_nbs = sb.read_notebooks(quarterly_folder)

quarterly_results = []
for path, scraps in quarterly_nbs.notebook_scraps.items():
    row = {
        'ticker': scraps['ticker'].data,
        'period': f"{scraps['start_date'].data[:7]}",  # Just YYYY-MM
        'return_pct': scraps['total_return_pct'].data,
        'volatility': scraps['avg_volatility'].data,
    }
    quarterly_results.append(row)

quarterly_df = pd.DataFrame(quarterly_results)
quarterly_df = quarterly_df.sort_values(['ticker', 'period'])
quarterly_df

In [ ]:
# Pivot to see returns by quarter for each stock
returns_pivot = quarterly_df.pivot(index='ticker', columns='period', values='return_pct')
print("Quarterly Returns (%)")
print(returns_pivot.round(2).to_string())

print("\nQuarterly Volatility")
vol_pivot = quarterly_df.pivot(index='ticker', columns='period', values='volatility')
print((vol_pivot * 100).round(1).to_string())

## 8. Accessing Full Data from Executed Notebooks

Remember, we also glued the full DataFrame with `sb.glue('prices', df, encoder='pandas')`. This means we can access detailed data from any executed notebook without re-running SQL queries.

In [ ]:
# Get the full price data from one of the notebooks
aapl_nb = sb.read_notebook(f"{OUTPUT_FOLDER}/AAPL_analysis.ipynb")
aapl_prices = aapl_nb.scraps['prices'].data

print(f"Full AAPL data shape: {aapl_prices.shape}")
print(f"Columns: {aapl_prices.columns.tolist()}")
aapl_prices.head()

---

## Summary: The Papermill + Scrapbook Workflow

1. **Create a Template Notebook**
   - Add a cell tagged with `parameters` containing default values
   - Use `sb.glue()` to export key results

2. **Run at Scale with Papermill**
   - `pm.execute_notebook(input, output, parameters={...})`
   - Loop over stocks, dates, or any parameter combinations

3. **Collect Results with Scrapbook**
   - `sb.read_notebook(path)` for a single notebook
   - `sb.read_notebooks(folder)` for many notebooks
   - Access data via `nb.scraps['name'].data`

4. **Aggregate and Analyze**
   - Build DataFrames from collected scraps
   - Create summary reports and visualizations

### When to Use This Pattern

- Running the same analysis across many securities
- Monthly/quarterly reports that need to be regenerated
- Backtesting with different parameters
- Any analysis that benefits from reproducible, parameterized execution

### Tips

- Add error handling when running many notebooks (some may fail)
- Use descriptive output filenames that include parameters
- Glue both summary metrics (easy to aggregate) and full data (for deep dives)
- Consider running notebooks in parallel for large jobs (using `concurrent.futures`)